# CS2 (국내반) --- Lecture 7 보충: `Robot` Class 다시 보기

*이 노트북은 채점/제출 대상이 아닙니다. 본 실습 노트북(`lecture07.ipynb`)을
풀기 전에 class와 object의 개념을 익숙한 Hubo 로봇으로 다시 확인해보는
용도입니다.*

## 왜 Colab용 `Robot`을 따로 만들었나?

CS1에서 사용한 `cs1robots`는 다음과 같은 계보를 가진 교육용 로봇이다.
- **Karel the Robot** (1981, Stanford, Richard Pattis): 격자 세계에서 로봇을
  움직이며 프로그래밍을 배우는 교육용 언어
- **RUR-PLE** (André Roberge): Karel을 Python으로 옮긴 학습 환경
- **cs1robots** (KAIST CS101, Otfried Cheong): RUR-PLE을 참고해서 만든
  Python 모듈. 로봇 이름이 KAIST의 휴머노이드 로봇 **Hubo**에서 왔다.
  ([원본 코드](https://github.com/otfried/cs101/blob/master/modules/cs1robots.py))
- KSA CS1 수업에서는 여기에 `turn_right` 등이 추가된 버전을 사용했다.

`cs1robots`는 로봇 세계를 **내 컴퓨터에 창(window)을 띄워서** 그린다. Colab은
원격 서버에서 실행되기 때문에 창을 띄울 화면이 없어서 `cs1robots`를 그대로
사용할 수 없다.

그래서 이 노트북에서는 `cs1robots`를 **Colab에서 동작하도록 간단하게 다시
만든 버전**을 사용한다.
- 사용법은 CS1 때와 같다: `load_world`, `create_world`, `Robot(beepers=6)`,
  `move()`, `turn_left()`, `turn_right()`, `front_is_clear()`,
  `right_is_clear()`, `on_beeper()`, `set_trace("blue")`, `set_pause(0.1)`, ...
  - CS1 Week11 실습 코드(`load_world("./worlds/maze.wld")` 등)를 그대로
    붙여넣어도 동작한다.
- state 변수 이름도 강의 슬라이드와 같다: `_x`, `_y`, `_dir`, `_beeper_bag`
- 그림은 창 대신 셀 아래에 나온다.
  - `set_pause(0.1)`처럼 pause를 설정하면 동작할 때마다 그림을 다시 그려서
    애니메이션처럼 보여준다.
  - pause가 없으면 원하는 시점에 `show_world()`를 호출해서 그림을 본다.

**아래 셀을 먼저 실행하자.** 이번에는 `import`해서 쓰는 대신 class 코드가
눈앞에 있으니, `class Robot` 부분을 위에서부터 천천히 읽어보자.

In [ ]:
# ==================================================================
# Colab용 cs1robots (간단 버전)
#  - KSA CS1에서 사용한 cs1robots와 같은 사용법으로 동작하도록 다시 만든 코드
#  - 원본(KAIST CS101, Otfried Cheong): https://github.com/otfried/cs101
#  - 창(window) 대신 matplotlib 그림으로 세계를 보여준다
# ==================================================================
import os, time, urllib.request
import matplotlib.pyplot as plt

try:
    from IPython.display import display, clear_output
    get_ipython()
    _IN_NOTEBOOK = True
except Exception:
    _IN_NOTEBOOK = False

_WORLD_URL = "https://raw.githubusercontent.com/SuminHan/book-cs2/main/notebooks/kor/worlds/"


class RobotError(Exception):
    pass


# ------------------------------------------------------------------
# 로봇 세계 (world)
#  - avenue(가로) x street(세로) 크기의 격자
#  - 벽은 (col, row) 좌표로 저장: avenue a, street s인 교차점은
#    (2a-1, 2s-1)이고, 두 교차점 사이(짝수 좌표)에 벽이 놓인다
# ------------------------------------------------------------------
class _World(object):
    def __init__(self, avenues=10, streets=10, walls=None, beepers=None, name=""):
        self.av = avenues
        self.st = streets
        self.walls = list(walls or [])
        self.beepers = dict(beepers or {})
        self.robots = []
        self.name = name

    def is_clear(self, col, row):
        if col <= 0 or row <= 0 or col >= 2 * self.av or row >= 2 * self.st:
            return False  # 세계의 가장자리(border)
        return (col, row) not in self.walls

    def add_beeper(self, av, st):
        self.beepers[(av, st)] = self.beepers.get((av, st), 0) + 1

    def remove_beeper(self, av, st):
        self.beepers[(av, st)] -= 1
        if self.beepers[(av, st)] == 0:
            del self.beepers[(av, st)]


_world = _World()


def create_world(avenues=10, streets=10):
    """벽이 없는 avenues x streets 크기의 세계를 새로 만든다."""
    global _world
    _world = _World(avenues, streets)


def load_world(filename):
    """.wld 파일에서 세계를 읽어온다.
    파일이 Colab에 없으면 book-cs2 저장소의 worlds 폴더에서 같은 이름의 파일을 받아온다."""
    global _world
    if os.path.exists(filename):
        with open(filename) as f:
            txt = f.read()
    else:
        url = _WORLD_URL + os.path.basename(filename)
        try:
            txt = urllib.request.urlopen(url).read().decode()
        except Exception:
            raise RuntimeError("world 파일을 찾을 수 없음: " + filename +
                               " (Colab 왼쪽 파일 탭에 업로드했는지 확인)")
    wd = {}
    exec(txt, {"__builtins__": {}}, wd)
    _world = _World(wd["avenues"], wd["streets"], wd.get("walls", []),
                    wd.get("beepers", {}), os.path.basename(filename))


def place_beepers(avenue, street, n=1):
    """(avenue, street) 위치에 beeper n개를 놓는다. (이 노트북에만 있는 함수)"""
    for i in range(n):
        _world.add_beeper(avenue, street)


# 방향: 0=N(북), 1=W(서), 2=S(남), 3=E(동)  -- turn_left 할 때마다 +1
_directions = [(0, 1), (-1, 0), (0, -1), (1, 0)]
_orient_dict = {"N": 0, "W": 1, "S": 2, "E": 3}


class Robot(object):
    # modifier: 새 로봇을 만들 때 자동으로 호출되어 state 변수들을 만든다
    def __init__(self, color="gray", orientation="E", beepers=0,
                 avenue=1, street=1):
        self._color = color
        self._dir = _orient_dict[orientation]  # 바라보는 방향
        self._x = avenue                       # 가로 위치 (avenue)
        self._y = street                       # 세로 위치 (street)
        self._beeper_bag = beepers             # 가지고 있는 beeper 수
        self._trace = None                     # 지나온 경로 (set_trace로 켬)
        self._trace_color = None
        self._delay = 0                        # set_pause로 설정
        _world.robots.append(self)

    # pure function: 로봇을 print 할 때 보이는 문자열
    def __str__(self):
        return ("Robot(" + self._color + ", (" + str(self._x) + "," +
                str(self._y) + "), " + "NWSE"[self._dir] +
                ", beepers=" + str(self._beeper_bag) + ")")

    # ---------------- 화면 관련 ----------------
    # modifier: 지나온 경로를 color 색으로 그리기 시작 (color 없으면 끔)
    def set_trace(self, color=None):
        if color:
            self._trace_color = color
            self._trace = [(self._x, self._y)]
        else:
            self._trace = None

    # modifier: 동작할 때마다 delay초 쉬면서 그림을 다시 그림 (애니메이션)
    def set_pause(self, delay=0):
        self._delay = delay

    def _refresh(self):
        if self._delay > 0:
            if _IN_NOTEBOOK:
                clear_output(wait=True)
                show_world()
            time.sleep(self._delay)

    # ---------------- 상태를 바꾸는 동작 (actuators) ----------------
    # modifier: 바라보는 방향으로 한 칸 이동
    def move(self):
        if self.front_is_clear():
            xx, yy = _directions[self._dir]
            self._x += xx
            self._y += yy
        else:
            raise RobotError("That move really hurt!\n Please, make sure that "
                             "there is no wall in front of me!")
        if self._trace is not None:
            self._trace.append((self._x, self._y))
        self._refresh()

    # modifier: 왼쪽으로 90도 회전
    def turn_left(self):
        self._dir = (self._dir + 1) % 4
        self._refresh()

    # modifier: 오른쪽으로 90도 회전
    def turn_right(self):
        self._dir = (self._dir - 1) % 4
        self._refresh()

    # modifier: 지금 칸의 beeper 하나를 주워 가방에 넣음
    def pick_beeper(self):
        if self.on_beeper():
            _world.remove_beeper(self._x, self._y)
            self._beeper_bag += 1
        else:
            raise RobotError("I must be on a beeper to pick it up.")
        self._refresh()

    # modifier: 가방의 beeper 하나를 지금 칸에 내려놓음
    def drop_beeper(self):
        if self.carries_beepers():
            self._beeper_bag -= 1
            _world.add_beeper(self._x, self._y)
        else:
            raise RobotError("I am not carrying any beepers.")
        self._refresh()

    # ---------------- 상태를 읽기만 하는 함수 (sensors) ----------------
    # pure function: 현재 위치를 (avenue, street)로 리턴
    def get_pos(self):
        return self._x, self._y

    def _is_clear(self, d):
        xx, yy = _directions[d]
        return _world.is_clear(2 * self._x - 1 + xx, 2 * self._y - 1 + yy)

    # pure function: 앞에 벽(또는 가장자리)이 없으면 True
    def front_is_clear(self):
        return self._is_clear(self._dir)

    # pure function: 왼쪽에 벽이 없으면 True
    def left_is_clear(self):
        return self._is_clear((self._dir + 1) % 4)

    # pure function: 오른쪽에 벽이 없으면 True
    def right_is_clear(self):
        return self._is_clear((self._dir - 1) % 4)

    # pure function: 북쪽을 바라보고 있으면 True
    def facing_north(self):
        return self._dir == 0

    # pure function: beeper를 하나라도 가지고 있으면 True
    def carries_beepers(self):
        return self._beeper_bag > 0

    # pure function: 지금 서 있는 칸에 beeper가 있으면 True
    def on_beeper(self):
        return (self._x, self._y) in _world.beepers

    # (실습 2) 여기에 새로운 함수를 추가해보자
    # ADD ADDITIONAL CODE HERE!


# ------------------------------------------------------------------
# 세계를 그림으로 보여주는 함수 (Robot class의 일부가 아님)
# ------------------------------------------------------------------
def show_world(title=None):
    W, H = _world.av, _world.st
    fig, ax = plt.subplots(figsize=(0.42 * W + 1, 0.42 * H + 1))
    for i in range(1, W + 1):
        ax.plot([i, i], [0.5, H + 0.5], color="#dddddd", zorder=0)
    for j in range(1, H + 1):
        ax.plot([0.5, W + 0.5], [j, j], color="#dddddd", zorder=0)
    ax.add_patch(plt.Rectangle((0.5, 0.5), W, H, fill=False,
                               edgecolor="darkred", linewidth=3))
    for (col, row) in _world.walls:
        if col % 2 == 0:   # 세로 벽
            x = col / 2 + 0.5
            ax.plot([x, x], [row / 2, row / 2 + 1], color="darkred", linewidth=3)
        else:              # 가로 벽
            y = row / 2 + 0.5
            ax.plot([col / 2, col / 2 + 1], [y, y], color="darkred", linewidth=3)
    for (x, y), n in _world.beepers.items():
        ax.add_patch(plt.Circle((x, y), 0.3, color="#2b6cb0", zorder=2))
        ax.text(x + 0.3, y + 0.3, str(n), color="#2b6cb0", ha="center",
                va="center", fontsize=9, fontweight="bold", zorder=5)
    arrows = {0: "^", 1: "<", 2: "v", 3: ">"}
    for r in _world.robots:
        if r._trace:
            ax.plot([p[0] for p in r._trace], [p[1] for p in r._trace],
                    color=r._trace_color, alpha=0.6, linewidth=2, zorder=1)
        ax.plot(r._x, r._y, marker=arrows[r._dir], markersize=13,
                color=r._color, markeredgecolor="black", zorder=4)
    ax.set_xlim(0.3, W + 0.7)
    ax.set_ylim(0.3, H + 0.7)
    ax.set_xticks(range(1, W + 1))
    ax.set_yticks(range(1, H + 1))
    ax.set_aspect("equal")
    ax.set_title(title if title is not None else ("Robot World: " + _world.name))
    plt.show()

print("cs1robots (Colab version) ready!")

## 1. `Robot` class 사용하기 (슬라이드 예제)

강의 슬라이드의 예제를 그대로 실행해보자.

- `hubo = Robot(beepers=6)`을 실행하면 `__init__(hubo, 6)`이 호출된다.
  - 즉, 새 객체 `hubo`가 `self` 자리에 들어가고, `__init__` 안의
    `self._beeper_bag = beepers`가 `hubo._beeper_bag = 6`이 된다.
- `hubo.move()`는 `move(hubo)`가 호출되는 것이다.

In [ ]:
create_world()

hubo = Robot(beepers=6)
# __init__(hubo, 6) invoked

for i in range(9):
    hubo.move()
    # move(hubo) invoked

amy = Robot(color="red", beepers=3)
# __init__(amy, 3) invoked

if amy.carries_beepers():
    # carries_beepers(amy) ..
    print("amy는 beeper를 가지고 있다")

print(hubo)
print(amy)
show_world("hubo (gray) and amy (red)")

## 2. `hubo.move()`는 정말 `move(hubo)`일까?

class 안에 정의된 함수는 `Robot.move`처럼 class 이름으로도 꺼낼 수 있다.
`hubo.move()`는 파이썬이 `Robot.move(hubo)`로 바꿔서 실행해주는 것이다.
그래서 **class 안의 모든 함수는 첫 번째 파라미터로 `self`를 받는다.**

아래 셀을 실행해서 두 방법의 결과가 같은지 확인해보자.

In [ ]:
create_world()
hubo = Robot(beepers=6)

hubo.move()          # 보통 쓰는 방법
print(hubo)          # Robot(gray, (2,1), E, beepers=6)

Robot.move(hubo)     # 파이썬이 실제로 하는 일
print(hubo)          # Robot(gray, (3,1), E, beepers=6)

## 3. Object와 Class

`Robot` class는 `hubo`, `amy`와 같은 로봇의 **type**을 정의한 것으로,
state 변수와 함수들로 구성된다:
- **state 변수**: `_x`, `_y`, `_dir`, `_beeper_bag` 등 각 로봇의 상태
  - `__init__` 함수에서 초기화/생성된다.
- **state 변수를 읽고/쓰는 함수**: `move`, `carries_beepers` 등
  - `move`는 state 변수를 **수정**하고 return 값이 없다 → **modifier**
  - `carries_beepers`는 state 변수를 **수정하지 않고** return 값이 있다
    → **pure function**

Object type을 **class**라고 부른다.
- `hubo`, `amy`는 **object**
- `Robot`은 **class**. 즉, `hubo`와 `amy`의 type

같은 class로 만든 object라도 **state 변수는 각자 따로** 가진다. 아래 셀에서
`hubo`만 움직였을 때 `amy`의 state는 어떻게 되는지 확인해보자.

In [ ]:
create_world()
hubo = Robot(beepers=6)
amy = Robot(color="red", beepers=3, avenue=1, street=3)

print(type(hubo))   # <class '__main__.Robot'>
print(type(amy))    # <class '__main__.Robot'>

hubo.move()
hubo.move()
hubo.drop_beeper()

# state 변수를 직접 들여다보기
print("hubo:", hubo._x, hubo._y, hubo._dir, hubo._beeper_bag)   # hubo: 3 1 3 5
print("amy :", amy._x, amy._y, amy._dir, amy._beeper_bag)       # amy : 1 3 3 3

show_world("only hubo moved")

## 4. CS1 코드 그대로 돌려보기

CS1 Week11에서 했던 "오른손으로 벽 짚고 세계 한 바퀴 돌기" 코드를 그대로
실행해보자.
- `load_world("./worlds/wall3.wld")`: 이 노트북의 `load_world`는 파일이 Colab에
  없으면 book-cs2 저장소에서 같은 이름의 world 파일을 받아온다.
  (`wall1.wld`, `wall2_1.wld`, `wall2_2.wld`, `wall3.wld`, `maze.wld`,
  `window1.wld`, `window2.wld`)
- 직접 만든 `.wld` 파일은 Colab 왼쪽의 파일(📁) 탭에 업로드해서 쓰면 된다.
- `set_pause(0.1)` 때문에 로봇이 움직이는 모습이 애니메이션으로 나온다.

In [ ]:
load_world("./worlds/wall3.wld")

hubo = Robot(beepers = 1)
hubo.set_trace("blue")
hubo.set_pause(0.1)

def go_around_world():
    hubo.drop_beeper()
    hubo.move()

    while not hubo.on_beeper():
        if hubo.right_is_clear():
            hubo.turn_right()
            hubo.move()
        elif hubo.front_is_clear():
            hubo.move()
        else:
            hubo.turn_left()

    hubo.pick_beeper()


go_around_world()

## 5. 실습

### 실습 1

새 세계를 만들고, `hubo`가 beeper 6개를 가지고 출발해서 동쪽으로 한 칸씩
이동하면서 **지나가는 칸마다 beeper를 1개씩 놓도록** 해보자. 가방의 beeper가
다 떨어지면 멈춘다. (`carries_beepers`를 이용)

In [ ]:
create_world()
hubo = Robot(beepers=6)

# ADD ADDITIONAL CODE HERE!


print(hubo)   # Robot(gray, (7,1), E, beepers=0)
show_world("Practice 1")

### 실습 2

맨 위의 `Robot` class 코드 셀에서 `# (실습 2) 여기에 새로운 함수를 추가해보자`
부분에 다음 두 함수를 **class 안에** 추가하고, 그 셀을 **다시 실행**한 뒤 아래
셀을 실행해보자.

1. `turn_around` (**modifier**)
   - 입력 파라미터: `self`
   - 동작: `self`를 180도 회전 (뒤로 돌기)
   - 리턴값: 없음
2. `beeper_count` (**pure function**)
   - 입력 파라미터: `self`
   - 리턴값: `self`가 가지고 있는 beeper의 수

*class 코드를 고친 뒤에는 class 셀을 다시 실행해야 바뀐 내용이 반영되고,
그 전에 만든 object가 아니라 **새로 만든 object**부터 새 함수를 쓸 수 있다.*

In [ ]:
create_world()
hubo = Robot(beepers=4)

hubo.turn_around()
print(hubo)                 # Robot(gray, (1,1), W, beepers=4)
hubo.turn_left()
hubo.turn_around()
print(hubo)                 # Robot(gray, (1,1), N, beepers=4)

hubo.drop_beeper()
print(hubo.beeper_count())  # 3

### 실습 3

`Robot` class의 각 함수가 **modifier**인지 **pure function**인지 표를
채워보자. (이 셀을 더블클릭해서 편집)

| 함수 | modifier / pure function |
|---|---|
| `__init__` | |
| `__str__` | |
| `move` | |
| `turn_left` | |
| `turn_right` | |
| `carries_beepers` | |
| `on_beeper` | |
| `front_is_clear` | |
| `right_is_clear` | |
| `get_pos` | |
| `facing_north` | |
| `drop_beeper` | |
| `pick_beeper` | |
| `turn_around` (실습 2) | |
| `beeper_count` (실습 2) | |

### 실습 4

beeper가 `(3,1)`, `(5,1)`, `(8,1)`에 놓여 있다. 빈 가방으로 출발한 `hubo`가
동쪽으로 끝까지 가면서 **beeper를 모두 줍도록** 해보자.
(`front_is_clear`, `on_beeper`, `pick_beeper`를 이용)

In [ ]:
create_world()
place_beepers(3, 1)
place_beepers(5, 1, 2)
place_beepers(8, 1)
hubo = Robot()

# ADD ADDITIONAL CODE HERE!


print(hubo)   # Robot(gray, (10,1), E, beepers=4)
show_world("Practice 4")